# Tutorial: FID-50K Generation for DuoDiT

Audience:
- DuoDiT/DiT users who need to generate 50,000 samples for FID or ADM-style evaluation.

Prerequisites:
- This notebook is run from the DuoDiT repository root.
- A CUDA environment with PyTorch, diffusers, timm, torchvision, PIL, tqdm, and numpy installed.
- A trained checkpoint path if evaluating your custom x2/PEFT model.
- Enough disk space for 50K PNGs plus one `.npz` file. At 256x256, budget roughly 10-20 GB depending on PNG compression and filesystem overhead.

Learning goals:
- Configure full-ImageNet or class-subset FID-50K sampling.
- Build the exact `torchrun sample_ddp.py` command.
- Run a small sanity sample before the expensive 50K run.
- Verify the generated folder and `.npz` output.
- Optionally compute FID against a matching real-image reference.


## Outline

1. Check the repo and GPU environment.
2. Configure model, checkpoint, classes, sampling steps, VAE, CFG, and batch size.
3. Build the sampling command.
4. Run a small sanity generation.
5. Run full FID-50K generation.
6. Validate generated outputs.
7. Optional: compute FID with `cleanfid` or ADM/guided-diffusion tools.


## Step 0 - How to use this notebook

Edit only the configuration cell first. The long-running cells are disabled by default through `RUN_SANITY_SAMPLE = False` and `RUN_FULL_FID50K = False`.

Recommended flow:

1. Set `CKPT` to your model checkpoint.
2. Set `CLASS_IDS` to the same class IDs used for training, or `None` for all 1000 ImageNet classes.
3. Run the check and command-building cells.
4. Flip `RUN_SANITY_SAMPLE` to `True` and generate a tiny sample.
5. Inspect the sanity output.
6. Flip `RUN_FULL_FID50K` to `True` for the real 50K generation.


In [ ]:
from __future__ import annotations

import os
import shlex
import subprocess
from pathlib import Path
from typing import Iterable

REPO_ROOT = Path.cwd().resolve()
print(f"Repo root: {REPO_ROOT}")

required_files = ["sample_ddp.py", "models.py", "download.py", "diffusion"]
missing = [name for name in required_files if not (REPO_ROOT / name).exists()]
if missing:
    raise FileNotFoundError(f"Run this notebook from the DuoDiT repo root. Missing: {missing}")

print("Repository check passed.")


## Step 1 - Check GPU and package availability

FID-50K generation uses `sample_ddp.py`, which requires CUDA and `torchrun`. This cell checks the basics without downloading model weights or launching sampling.


In [ ]:
import importlib.util
import shutil

packages = ["torch", "torchvision", "diffusers", "timm", "PIL", "numpy", "tqdm"]
for package in packages:
    print(f"{package:12s}: {'ok' if importlib.util.find_spec(package) else 'missing'}")

print(f"torchrun     : {shutil.which('torchrun') or 'missing'}")

import torch
print(f"torch        : {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
print(f"gpu count    : {torch.cuda.device_count()}")
for idx in range(torch.cuda.device_count()):
    print(f"gpu {idx:<2d}      : {torch.cuda.get_device_name(idx)}")


## Step 2 - Configure the FID-50K run

Important choices:

- `CKPT`: set this to your custom checkpoint. Leave it empty only when you want the pretrained DiT baseline.
- `CLASS_IDS`: use `None` for full ImageNet. Use a list for a class-subset model, for example `[972, 973, 974, 975, 976]`.
- `VAE`: use `"mse"` for the standard DiT paper-style FID setting; use `"ema"` only if you intentionally want that decoder.
- `CFG_SCALE`: use `1.0` for unguided FID comparisons unless your experiment explicitly reports guided FID.
- `NUM_SAMPLING_STEPS`: standard DiT FID runs use 250 DDPM steps.


In [ ]:
import torch

# Model and checkpoint
MODEL = "DiT-XL/2"
IMAGE_SIZE = 256
NUM_CLASSES = 1000

# Set this for your model. Examples:
# CKPT = "results/checkpoints/003-DiT-XL-2-x2-finetune/checkpoints/epoch-0-loss-0.1425.pt"
# CKPT = "DiT-XL-2-256x256.pt"  # pretrained baseline, auto-downloaded if needed
CKPT = ""

# Use None for full ImageNet FID-50K. Use a list for class-subset evaluation.
# Example: CLASS_IDS = [972, 973, 974, 975, 976]
CLASS_IDS = None

# Sampling recipe
NUM_FID_SAMPLES = 50_000
SANITY_NUM_SAMPLES = 32 # SANITY_NUM_SAMPLES is the small test sample count used before running the expensive full FID-50K generation
NUM_SAMPLING_STEPS = 250
VAE = "mse"
CFG_SCALE = 1.0
GLOBAL_SEED = 0
TF32 = True

# Hardware and output
NUM_GPUS = max(1, torch.cuda.device_count())
PER_PROC_BATCH_SIZE = 16
SANITY_SAMPLE_DIR = "fid50k_sanity_samples"
FID50K_SAMPLE_DIR = "fid50k_samples"

# Safety flags. Flip these manually when ready.
RUN_SANITY_SAMPLE = False
RUN_FULL_FID50K = False


## Step 3 - Validate the configuration

This catches common mistakes before a long run starts: missing checkpoint paths, invalid class IDs, invalid VAE choice, missing GPUs, and impossible batch settings.


In [ ]:
def normalize_classes(class_ids: Iterable[int] | None, num_classes: int) -> list[int] | None:
    if class_ids is None:
        return None
    classes = sorted(set(int(cls) for cls in class_ids))
    if not classes:
        raise ValueError("CLASS_IDS must be None or a non-empty list of class IDs.")
    invalid = [cls for cls in classes if cls < 0 or cls >= num_classes]
    if invalid:
        raise ValueError(f"Invalid class IDs for NUM_CLASSES={num_classes}: {invalid}")
    return classes

CLASS_IDS = normalize_classes(CLASS_IDS, NUM_CLASSES)

if MODEL != "DiT-XL/2" and not CKPT:
    raise ValueError("Custom model sizes require CKPT to be set.")
if IMAGE_SIZE not in {256, 512}:
    raise ValueError("IMAGE_SIZE must be 256 or 512 for this repo's sampler.")
if VAE not in {"mse", "ema"}:
    raise ValueError("VAE must be 'mse' or 'ema'.")
if CFG_SCALE < 1.0:
    raise ValueError("CFG_SCALE should be >= 1.0.")
if NUM_GPUS < 1:
    raise ValueError("NUM_GPUS must be at least 1.")
if PER_PROC_BATCH_SIZE < 1:
    raise ValueError("PER_PROC_BATCH_SIZE must be at least 1.")
if not torch.cuda.is_available():
    raise RuntimeError("sample_ddp.py requires CUDA. Use a GPU runtime before running sampling.")

if CKPT:
    ckpt_path = Path(CKPT)
    known_pretrained = {"DiT-XL-2-256x256.pt", "DiT-XL-2-512x512.pt"}
    if CKPT not in known_pretrained and not ckpt_path.exists():
        raise FileNotFoundError(f"CKPT does not exist: {CKPT}")
else:
    print("CKPT is empty: the sampler will use the pretrained DiT-XL/2 checkpoint for the selected IMAGE_SIZE.")

print("Configuration is valid.")
print(f"Classes: {'all ImageNet classes' if CLASS_IDS is None else CLASS_IDS}")
print(f"Global batch size: {NUM_GPUS * PER_PROC_BATCH_SIZE}")


## Step 4 - Build the sampling command

The same function builds both the tiny sanity command and the full FID-50K command. It passes `--classes` only when `CLASS_IDS` is not `None`.


In [ ]:
def shell_join(cmd: list[str]) -> str:
    return " ".join(shlex.quote(str(part)) for part in cmd)


def class_label(classes: list[int] | None, num_classes: int) -> str:
    if classes is None:
        return "all"
    if len(classes) == num_classes and classes == list(range(num_classes)):
        return "all"
    if len(classes) <= 8:
        return "-".join(str(cls) for cls in classes)
    return f"{classes[0]}-{classes[-1]}-n{len(classes)}"


def ckpt_label(ckpt: str) -> str:
    return Path(ckpt).name.replace(".pt", "") if ckpt else "pretrained"


def expected_output_paths(sample_dir: str) -> tuple[Path, Path]:
    model_name = MODEL.replace("/", "-")
    folder_name = (
        f"{model_name}-{ckpt_label(CKPT)}-size-{IMAGE_SIZE}-vae-{VAE}-"
        f"cfg-{CFG_SCALE}-classes-{class_label(CLASS_IDS, NUM_CLASSES)}-seed-{GLOBAL_SEED}"
    )
    sample_folder = Path(sample_dir) / folder_name
    npz_path = Path(str(sample_folder) + ".npz")
    return sample_folder, npz_path


def build_sample_command(num_samples: int, sample_dir: str) -> list[str]:
    cmd = [
        "torchrun",
        "--standalone",
        "--nnodes=1",
        f"--nproc_per_node={NUM_GPUS}",
        "sample_ddp.py",
        "--model", MODEL,
        "--sample-dir", sample_dir,
        "--per-proc-batch-size", str(PER_PROC_BATCH_SIZE),
        "--num-fid-samples", str(num_samples),
        "--image-size", str(IMAGE_SIZE),
        "--num-classes", str(NUM_CLASSES),
        "--cfg-scale", str(CFG_SCALE),
        "--num-sampling-steps", str(NUM_SAMPLING_STEPS),
        "--vae", VAE,
        "--global-seed", str(GLOBAL_SEED),
    ]
    if not TF32:
        cmd.append("--no-tf32")
    if CKPT:
        cmd.extend(["--ckpt", CKPT])
    if CLASS_IDS is not None:
        cmd.append("--classes")
        cmd.extend(str(cls) for cls in CLASS_IDS)
    return cmd

sanity_cmd = build_sample_command(SANITY_NUM_SAMPLES, SANITY_SAMPLE_DIR)
fid50k_cmd = build_sample_command(NUM_FID_SAMPLES, FID50K_SAMPLE_DIR)
sanity_sample_folder, sanity_npz_path = expected_output_paths(SANITY_SAMPLE_DIR)
fid50k_sample_folder, fid50k_npz_path = expected_output_paths(FID50K_SAMPLE_DIR)

print("Sanity command:")
print(shell_join(sanity_cmd))
print("\nFID-50K command:")
print(shell_join(fid50k_cmd))
print("\nExpected outputs:")
print("Sanity outputs:")
print(f"PNG folder: {sanity_sample_folder}")
print(f"NPZ file  : {sanity_npz_path}")
print("\nFID-50K outputs:")
print(f"PNG folder: {fid50k_sample_folder}")
print(f"NPZ file  : {fid50k_npz_path}")


## Step 5 - Run a tiny sanity sample

This should complete much faster than the 50K run. Use it to catch checkpoint loading, package, VAE download/cache, CUDA, and shape errors.

Set `RUN_SANITY_SAMPLE = True` in the configuration cell before running this cell.


In [ ]:
if RUN_SANITY_SAMPLE:
    print(shell_join(sanity_cmd))
    subprocess.run(sanity_cmd, cwd=REPO_ROOT, check=True)
else:
    print("Sanity run skipped. Set RUN_SANITY_SAMPLE = True in Step 2 to launch it.")


## Step 6 - Inspect the sanity output

After a sanity run, check that the folder contains numbered PNG files and that the `.npz` file was created. For tiny sanity runs, it is safe to load the `.npz` into memory.


In [ ]:
def summarize_outputs(folder: Path, npz: Path, load_small_npz: bool = True) -> None:
    print(f"Folder exists: {folder.exists()} -> {folder}")
    if folder.exists():
        pngs = sorted(folder.glob("*.png"))
        print(f"PNG count: {len(pngs)}")
        if pngs[:3]:
            print("First PNGs:", [p.name for p in pngs[:3]])
    print(f"NPZ exists   : {npz.exists()} -> {npz}")
    if npz.exists():
        size_gb = npz.stat().st_size / 1024**3
        print(f"NPZ size    : {size_gb:.2f} GB")
        if load_small_npz and size_gb < 1:
            import numpy as np
            arr = np.load(npz)["arr_0"]
            print(f"NPZ arr_0   : shape={arr.shape}, dtype={arr.dtype}, min={arr.min()}, max={arr.max()}")
        elif load_small_npz:
            print("NPZ is large; skipped loading it into memory.")

summarize_outputs(sanity_sample_folder, sanity_npz_path)


## Step 7 - Run full FID-50K generation

This is the expensive cell. It generates 50,000 PNG files and then packs exactly 50,000 samples into an ADM-style `.npz` file with key `arr_0` and shape `(50000, H, W, 3)`.

Before running:

- Confirm `CKPT` points to the intended checkpoint.
- Confirm `CLASS_IDS` matches the intended evaluation distribution.
- Use `VAE = "mse"`, `CFG_SCALE = 1.0`, and `NUM_SAMPLING_STEPS = 250` for the standard DiT FID setting.
- Make sure `FID50K_SAMPLE_DIR` has enough free disk space.

Set `RUN_FULL_FID50K = True` in the configuration cell before running this cell.


In [ ]:
if RUN_FULL_FID50K:
    print(shell_join(fid50k_cmd))
    subprocess.run(fid50k_cmd, cwd=REPO_ROOT, check=True)
else:
    print("Full FID-50K run skipped. Set RUN_FULL_FID50K = True in Step 2 to launch it.")


## Step 8 - Validate the FID-50K output

For a completed 50K run, this checks the PNG count and `.npz` file. It avoids loading a large 50K `.npz` into memory.


In [ ]:
summarize_outputs(fid50k_sample_folder, fid50k_npz_path, load_small_npz=False)

if fid50k_sample_folder.exists():
    png_count = len(list(fid50k_sample_folder.glob("*.png")))
    if png_count >= NUM_FID_SAMPLES:
        print(f"OK: found at least {NUM_FID_SAMPLES} PNG files.")
    else:
        print(f"Incomplete: found {png_count} PNG files, expected {NUM_FID_SAMPLES}.")

if fid50k_npz_path.exists():
    print("OK: ADM-style NPZ file exists.")
else:
    print("Missing NPZ. The sampler may still be running, failed, or did not reach rank-0 NPZ creation.")


## Step 9 - Optional FID with clean-fid

Use this when you want folder-to-folder FID. The generated folder must be compared against the matching real-image distribution:

- Full ImageNet generation: compare against all ImageNet validation images at the same resolution/crop convention.
- Class-subset generation: compare only against the same real classes.

`cleanfid` is not installed automatically here. Install it in your environment if needed.


In [ ]:
REAL_IMAGES_DIR = ""  # Example: "/path/to/imagenet/val_subset_972_976"
RUN_CLEANFID = False

if RUN_CLEANFID:
    if not REAL_IMAGES_DIR:
        raise ValueError("Set REAL_IMAGES_DIR before running clean-fid.")
    from cleanfid import fid
    score = fid.compute_fid(str(fid50k_sample_folder), REAL_IMAGES_DIR)
    print(f"FID: {score:.4f}")
else:
    print("clean-fid skipped. Set REAL_IMAGES_DIR and RUN_CLEANFID = True to compute it.")


## Step 10 - Optional ADM/guided-diffusion evaluation

The `.npz` file created by `sample_ddp.py` follows the ADM/guided-diffusion convention: `arr_0` is a uint8 array of generated RGB images.

This cell bootstraps the official OpenAI evaluator when needed:

- If `external/guided-diffusion` does not exist, it clones `https://github.com/openai/guided-diffusion.git`.
- If the ImageNet reference `.npz` for `IMAGE_SIZE` does not exist, it downloads it from OpenAI's public reference batches.
- Then it runs `evaluations/evaluator.py reference.npz generated.npz`.

For class-subset experiments, the official ImageNet reference batch is full ImageNet, so it is not a matched subset reference. Use this ADM cell for full ImageNet evaluation unless you have built a subset-specific reference `.npz`.


In [ ]:
ADM_REPO_DIR = REPO_ROOT / "external" / "guided-diffusion"
ADM_EVALUATOR = ADM_REPO_DIR / "evaluations" / "evaluator.py"
REFERENCE_DIR = REPO_ROOT / "fid50k_references"

IMAGENET_REFERENCE_URLS = {
    256: "https://openaipublic.blob.core.windows.net/diffusion/jul-2021/ref_batches/imagenet/256/VIRTUAL_imagenet256_labeled.npz",
    512: "https://openaipublic.blob.core.windows.net/diffusion/jul-2021/ref_batches/imagenet/512/VIRTUAL_imagenet512.npz",
}

RUN_ADM_EVAL = False
ALLOW_FULL_IMAGENET_REFERENCE_FOR_SUBSET = False


def download_file(url: str, destination: Path) -> None:
    import urllib.request

    destination.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = destination.with_suffix(destination.suffix + ".tmp")
    print(f"Downloading {url}")
    print(f"       -> {destination}")
    urllib.request.urlretrieve(url, tmp_path)
    tmp_path.replace(destination)


def ensure_adm_evaluator() -> Path:
    if ADM_EVALUATOR.exists():
        return ADM_EVALUATOR
    if not ADM_REPO_DIR.exists():
        ADM_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        clone_cmd = ["git", "clone", "https://github.com/openai/guided-diffusion.git", str(ADM_REPO_DIR)]
        print(shell_join(clone_cmd))
        subprocess.run(clone_cmd, check=True)
    if not ADM_EVALUATOR.exists():
        raise FileNotFoundError(f"ADM evaluator was not found after setup: {ADM_EVALUATOR}")
    return ADM_EVALUATOR


def ensure_reference_npz() -> Path:
    if IMAGE_SIZE not in IMAGENET_REFERENCE_URLS:
        raise ValueError(f"No default ADM ImageNet reference URL configured for IMAGE_SIZE={IMAGE_SIZE}.")
    url = IMAGENET_REFERENCE_URLS[IMAGE_SIZE]
    reference_npz = REFERENCE_DIR / Path(url).name
    if not reference_npz.exists():
        download_file(url, reference_npz)
    return reference_npz


if RUN_ADM_EVAL:
    if CLASS_IDS is not None and not ALLOW_FULL_IMAGENET_REFERENCE_FOR_SUBSET:
        raise ValueError(
            "CLASS_IDS is set, but the default ADM reference is full ImageNet. "
            "Use a subset-specific reference .npz, or set "
            "ALLOW_FULL_IMAGENET_REFERENCE_FOR_SUBSET = True if you intentionally want the mismatched full-reference run."
        )
    if not fid50k_npz_path.exists():
        raise FileNotFoundError(f"Generated NPZ not found: {fid50k_npz_path}")

    adm_evaluator = ensure_adm_evaluator()
    reference_npz = ensure_reference_npz()
    adm_cmd = ["python", str(adm_evaluator), str(reference_npz), str(fid50k_npz_path.resolve())]
    print(shell_join(adm_cmd))
    subprocess.run(adm_cmd, check=True, cwd=ADM_REPO_DIR / "evaluations")
else:
    print("ADM evaluator skipped. Set RUN_ADM_EVAL = True to clone/download missing assets and run it.")
    print(f"Evaluator path : {ADM_EVALUATOR}")
    print(f"Reference dir  : {REFERENCE_DIR}")


## Common pitfalls

- Sampling all 1000 classes for a model trained on a class subset will give a mismatched FID setup. Use `CLASS_IDS` for subset experiments.
- Comparing subset samples against full ImageNet validation is also mismatched. The real-image reference must use the same class distribution.
- Using `CFG_SCALE > 1.0` changes the metric setting. Report it clearly if you use guided FID.
- `sample_ddp.py` does not resume partially completed folders. Use a fresh `FID50K_SAMPLE_DIR`, seed, or checkpoint label for clean runs.
- The first run may download pretrained DiT or VAE weights, so it can fail if the environment has no network and no cached weights.


## Exercise

Create two commands without running them:

1. A full ImageNet pretrained DiT baseline using `CKPT = ""` and `CLASS_IDS = None`.
2. Your custom checkpoint over a class subset using `CLASS_IDS = [972, 973, 974, 975, 976]`.

Compare the printed output folder names. The class-subset command should include `--classes` and the output folder should include the class label.
